In [3]:
# 3) Check uploaded Excel files
from pathlib import Path

print('Excel files in /content:')
for p in sorted(Path('/content').glob('*.xlsx')):
    print('-', p.name)

# ============================================================
# CNC OVERTIME MINIMIZATION EXACT MILP
# Gurobi + Integer Job Splitting + Global Machine No-Overlap + HTML Gantt
# ------------------------------------------------------------
# Required files in /content:
#   1) 492-güncel sevkiyat*.xlsx or any *sevkiyat*.xlsx
#   2) SDST*.xlsx
#   3) machine_group_data*.xlsx
#
# Preserved model logic from the heuristic version:
#   - 2 operations: Op10 and Op20
#   - Each job may be split into integer split lots up to MAX_SPLITS
#   - q_{jr} integer, sum_r q_{jr} = original quantity
#   - Op20 cannot start before the same split part's Op10 is completed
#   - Op20 is assigned to the same machine group selected for Op10
#   - Physical machines are shared resources: Op10 and Op20 cannot overlap on the same machine
#   - Sequence-dependent setup time is based on diameter transitions
#   - First task on a machine has INITIAL_SETUP_MIN
#   - Overtime is measured as overlap of PROCESSING interval with 17:00-21:00 windows
#   - Optional rule: do not start a new operation inside 17:00-21:00
#   - Tardiness limit is enforced at job level by latest Op20 completion among split parts
#   - Outputs: exact schedule Excel, verification Excel, HTML Gantt
# ============================================================

import os
import math
import re
from dataclasses import dataclass
from datetime import datetime, timedelta, time
from collections import defaultdict

import pandas as pd
import numpy as np
!pip install psutil
from gurobipy import Model, GRB, quicksum

import plotly.graph_objects as go

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# ============================================================
# 0. CONFIGURATION
# ============================================================

BASE_DIR = Path('C:\\Users\\emrec\\Documents\\parallel_scheduling\\exact1')


def find_input_file(patterns, description):
    """Finds the first matching uploaded file in BASE_DIR."""
    for pattern in patterns:
        matches = sorted(BASE_DIR.glob(pattern))
        if matches:
            return matches[0]
    available = "\n".join([p.name for p in sorted(BASE_DIR.glob('*.xlsx'))])
    raise FileNotFoundError(
        f"Could not find {description}.\n"
        f"Searched patterns: {patterns}\n"
        f"Available Excel files in /content:\n{available}"
    )


SHIPMENT_FILE = find_input_file(
    ['492-güncel sevkiyat*.xlsx', '492*güncel*sevkiyat*.xlsx', '*sevkiyat*.xlsx'],
    'shipment file'
)
SDST_FILE = find_input_file(['SDST*.xlsx', '*SDST*.xlsx'], 'SDST file')
MACHINE_GROUP_FILE = find_input_file(
    ['machine_group_data*.xlsx', '*machine*group*.xlsx'],
    'machine group file'
)

OUTPUT_SCHEDULE_FILE = BASE_DIR / 'optimized_exact_gurobi_schedule_split.xlsx'
OUTPUT_VALIDATION_FILE = BASE_DIR / 'verification_validation_exact_gurobi_split.xlsx'
OUTPUT_GANTT_FILE = BASE_DIR / 'exact_gurobi_gantt_split.html'

print('\nUsing files:')
print('SHIPMENT_FILE      =', SHIPMENT_FILE)
print('SDST_FILE          =', SDST_FILE)
print('MACHINE_GROUP_FILE =', MACHINE_GROUP_FILE)
print('OUTPUT_SCHEDULE    =', OUTPUT_SCHEDULE_FILE)
print('OUTPUT_VALIDATION  =', OUTPUT_VALIDATION_FILE)
print('OUTPUT_GANTT_HTML  =', OUTPUT_GANTT_FILE)

# Planning assumptions
PLANNING_START_HOUR = 7
DUE_TIME_HOUR = 17

# Overtime zone according to shipment Gantt logic
OVERTIME_START = time(17, 0)
OVERTIME_END = time(21, 0)

# If True, no new operation is allowed to START inside 17:00-21:00.
# A task may still continue into the window; that processing overlap is counted as OT.
DISALLOW_START_IN_OVERTIME = True

# Tardiness setting
TARDINESS_LIMIT_DAYS = 1.5
TARDINESS_LIMIT_MIN = TARDINESS_LIMIT_DAYS * 24 * 60

# Split settings
ALLOW_JOB_SPLITTING = True
MAX_SPLITS = 2
MIN_SPLIT_QTY = 100

# Setup settings
INITIAL_SETUP_MIN = 10.0
SAME_DIAM_SETUP_MIN = 5.0
DEFAULT_DIFF_DIAM_SETUP_MIN = 20.0

# Objective weights
# Primary target remains total overtime. Small tie-breakers make the MILP return compact schedules.
W_OVERTIME = 1.0
W_MAKESPAN = 0.003
W_SETUP = 0.01

# Solver settings
MIP_GAP = 0.05
TIME_LIMIT_SEC = 21600  # e.g., 3600. Use None for no limit.
OUTPUT_FLAG = 1

BIG_M = 10**7
EPS = 1e-5


# ============================================================
# 1. HELPER FUNCTIONS
# ============================================================

def normalize_machine_name(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    s = s.replace(' ', '')
    s = s.replace(',', '.')
    s = s.replace('O', '0')
    s = s.replace('T3.', 'T.3.')
    if re.match(r'^T3\d+', s):
        s = s.replace('T3', 'T.3.', 1)
    return s


def excel_serial_to_datetime_date(x):
    if pd.isna(x):
        return None
    if isinstance(x, datetime):
        return x.date()
    if isinstance(x, (int, float, np.integer, np.floating)):
        return (datetime(1899, 12, 30) + timedelta(days=float(x))).date()
    parsed = pd.to_datetime(x, errors='coerce')
    if pd.isna(parsed):
        return None
    return parsed.date()


def excel_time_to_timedelta(x):
    if pd.isna(x):
        return None
    if isinstance(x, timedelta):
        return x
    if isinstance(x, datetime):
        return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, time):
        return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, (int, float, np.integer, np.floating)):
        return timedelta(days=float(x))
    parsed = pd.to_datetime(str(x), errors='coerce')
    if pd.isna(parsed):
        return None
    return timedelta(hours=parsed.hour, minutes=parsed.minute, seconds=parsed.second)


def combine_excel_date_time(date_value, time_value):
    d = excel_serial_to_datetime_date(date_value)
    td = excel_time_to_timedelta(time_value)
    if d is None or td is None:
        return None
    return datetime.combine(d, time(0, 0)) + td


def minutes_between(start_dt, end_dt):
    if end_dt <= start_dt:
        end_dt = end_dt + timedelta(days=1)
    return (end_dt - start_dt).total_seconds() / 60.0


def overlap_minutes(a_start, a_end, b_start, b_end):
    latest_start = max(a_start, b_start)
    earliest_end = min(a_end, b_end)
    return max(0.0, (earliest_end - latest_start).total_seconds() / 60.0)


def overtime_overlap_minutes(start_dt, end_dt):
    if end_dt <= start_dt:
        return 0.0
    total = 0.0
    current_day = start_dt.date()
    last_day = end_dt.date()
    while current_day <= last_day:
        ot_start = datetime.combine(current_day, OVERTIME_START)
        ot_end = datetime.combine(current_day, OVERTIME_END)
        total += overlap_minutes(start_dt, end_dt, ot_start, ot_end)
        current_day = current_day + timedelta(days=1)
    return total


def parse_overtime_text(value):
    if pd.isna(value):
        return 0.0
    s = str(value).strip().lower()
    if not s:
        return 0.0
    hours = 0
    minutes = 0
    h = re.search(r'(\d+)\s*saat', s)
    m = re.search(r'(\d+)\s*dk', s)
    if h:
        hours = int(h.group(1))
    if m:
        minutes = int(m.group(1))
    if not h and not m:
        nums = re.findall(r'\d+', s)
        if nums:
            minutes = int(nums[0])
    return float(hours * 60 + minutes)


def mode_or_first(series, default=None):
    s = series.dropna()
    if len(s) == 0:
        return default
    m = s.mode()
    if len(m) > 0:
        return m.iloc[0]
    return s.iloc[0]


def dt_to_min(dt, planning_start):
    return (dt - planning_start).total_seconds() / 60.0


def min_to_dt(minutes, planning_start):
    return planning_start + timedelta(minutes=float(minutes))


# ============================================================
# 2. DATA CLASSES
# ============================================================

@dataclass
class Job:
    job_id: str
    part_no: str
    due_date: datetime
    quantity: int
    diameter: float
    eligible_groups_op10: list
    eligible_groups_op20: list


# ============================================================
# 3. LOAD INPUT DATA
# ============================================================

def load_shipment_operations(path):
    raw = pd.read_excel(path, sheet_name=0, header=5, usecols='D:N', engine='openpyxl')
    raw.columns = [str(c).strip().replace(':', '') for c in raw.columns]
    raw = raw.dropna(how='all')

    rename_map = {
        'Tarih': 'date',
        'Başlangıç Saat': 'start_time',
        'Bitiş Saat': 'finish_time',
        'Parça No': 'part_no',
        'Makine No': 'machine',
        'CNC-1 operasyonu(piston)': 'op10_flag',
        'CNC-2 operasyonu (saplama)': 'op20_flag',
        'Adet': 'quantity',
        'Çap': 'diameter',
        'Makine Grubu': 'machine_group',
        'Fazla mesai': 'overtime_text',
    }
    raw = raw.rename(columns=rename_map)

    required = ['date', 'start_time', 'finish_time', 'part_no', 'machine', 'quantity', 'diameter', 'machine_group']
    for col in required:
        if col not in raw.columns:
            raise ValueError(f'Missing expected column in shipment operation sheet: {col}')

    raw = raw[raw['part_no'].notna()].copy()
    raw['machine'] = raw['machine'].apply(normalize_machine_name)
    raw['part_no'] = raw['part_no'].astype(int).astype(str)
    raw['quantity'] = pd.to_numeric(raw['quantity'], errors='coerce').fillna(0).astype(int)
    raw['diameter'] = pd.to_numeric(raw['diameter'], errors='coerce')
    raw['machine_group'] = pd.to_numeric(raw['machine_group'], errors='coerce').astype('Int64')

    raw['operation'] = np.where(raw.get('op10_flag').notna(), 10, np.where(raw.get('op20_flag').notna(), 20, np.nan))
    raw = raw[raw['operation'].notna()].copy()
    raw['operation'] = raw['operation'].astype(int)

    raw['start_dt'] = [combine_excel_date_time(d, t) for d, t in zip(raw['date'], raw['start_time'])]
    raw['finish_dt'] = [combine_excel_date_time(d, t) for d, t in zip(raw['date'], raw['finish_time'])]
    raw['duration_min'] = [minutes_between(s, f) for s, f in zip(raw['start_dt'], raw['finish_dt'])]
    raw['unit_min'] = raw['duration_min'] / raw['quantity'].replace(0, np.nan)
    return raw


def load_orders_from_sayfa2(path, use_shipped_quantity=False):
    df = pd.read_excel(path, sheet_name=1, header=None, engine='openpyxl')
    jobs = []
    current_date = None
    in_block = False

    for _, row in df.iterrows():
        first_values = row.dropna().tolist()
        if len(first_values) == 0:
            continue

        possible_date = None
        for val in row.tolist():
            if pd.isna(val):
                continue
            if isinstance(val, (int, float, np.integer, np.floating)) and 40000 < float(val) < 60000:
                possible_date = excel_serial_to_datetime_date(val)
                break
            if isinstance(val, datetime):
                possible_date = val.date()
                break

        if possible_date is not None:
            current_date = possible_date
            in_block = False
            continue

        row_text = ' '.join([str(v).lower() for v in row.dropna().tolist()])
        if 'sipariş' in row_text and 'adet' in row_text:
            in_block = True
            continue

        if in_block and current_date is not None:
            part = row.iloc[4] if len(row) > 4 else None
            order_qty = row.iloc[5] if len(row) > 5 else None
            shipped_qty = row.iloc[6] if len(row) > 6 else None

            if pd.isna(part) or pd.isna(order_qty):
                continue
            try:
                part_no = str(int(part))
            except Exception:
                continue

            qty_source = shipped_qty if use_shipped_quantity and not pd.isna(shipped_qty) else order_qty
            qty = int(round(float(qty_source)))
            if qty <= 0:
                continue

            due_dt = datetime.combine(current_date, time(DUE_TIME_HOUR, 0))
            job_id = f'{current_date.isoformat()}__{part_no}'
            jobs.append({
                'job_id': job_id,
                'part_no': part_no,
                'due_date': due_dt,
                'quantity': qty,
            })

    orders = pd.DataFrame(jobs)
    if orders.empty:
        raise ValueError('Could not parse any order rows from Sayfa2.')

    # CRITICAL FIX:
    # Sayfa2 sometimes contains more than one row for the same date + part.
    # Since job_id = date + part_no, those repeated rows create duplicate
    # Gurobi variable keys such as active[job_id, r]. Gurobi addVars() does
    # not allow duplicate keys. Therefore, duplicate job_id rows are aggregated
    # before Job objects and model variables are created.
    before_n = len(orders)
    orders = (
        orders.groupby(['job_id', 'part_no'], as_index=False)
        .agg(
            quantity=('quantity', 'sum'),
            due_date=('due_date', 'min')
        )
    )
    after_n = len(orders)
    if before_n != after_n:
        print(f'Duplicate order rows aggregated: {before_n} rows -> {after_n} unique jobs')

    return orders


def load_machine_groups(path):
    mg = pd.read_excel(path, sheet_name=0, engine='openpyxl')
    mg.columns = [str(c).strip() for c in mg.columns]
    mg = mg.dropna(subset=['Machine_number', 'Group']).copy()
    mg['Machine_number'] = mg['Machine_number'].apply(normalize_machine_name)
    mg['Group'] = pd.to_numeric(mg['Group'], errors='coerce').astype(int)

    group_to_machines = defaultdict(list)
    machine_to_group = {}
    for _, r in mg.iterrows():
        m = r['Machine_number']
        g = int(r['Group'])
        group_to_machines[g].append(m)
        machine_to_group[m] = g
    return dict(group_to_machines), machine_to_group


def load_sdst(path):
    sdst = pd.read_excel(path, sheet_name=0, engine='openpyxl')
    sdst.columns = [str(c).strip() for c in sdst.columns]
    setup = {}
    for _, r in sdst.dropna(subset=['diam_from', 'to_diam', 'setup_time']).iterrows():
        d1 = float(r['diam_from'])
        d2 = float(r['to_diam'])
        setup[(d1, d2)] = float(r['setup_time'])
    return setup


# ============================================================
# 4. BUILD MODEL DATA FROM FILES
# ============================================================

def build_problem_data(shipment_file, sdst_file, machine_group_file, use_shipped_quantity=False):
    ops = load_shipment_operations(shipment_file)
    orders = load_orders_from_sayfa2(shipment_file, use_shipped_quantity=use_shipped_quantity)
    group_to_machines, machine_to_group = load_machine_groups(machine_group_file)
    setup_dict = load_sdst(sdst_file)

    part_diam = ops.groupby('part_no')['diameter'].agg(lambda x: mode_or_first(x, default=0)).to_dict()

    eligible = defaultdict(lambda: defaultdict(set))
    for _, r in ops.iterrows():
        eligible[r['part_no']][int(r['operation'])].add(int(r['machine_group']))

    unit_time = {}
    grouped = ops.dropna(subset=['unit_min', 'machine_group']).groupby(['part_no', 'operation', 'machine_group'])
    for key, g in grouped:
        unit_time[(str(key[0]), int(key[1]), int(key[2]))] = float(g['unit_min'].median())

    fallback_part_op = ops.dropna(subset=['unit_min']).groupby(['part_no', 'operation'])['unit_min'].median().to_dict()
    fallback_op = ops.dropna(subset=['unit_min']).groupby(['operation'])['unit_min'].median().to_dict()
    global_unit = float(ops['unit_min'].dropna().median())

    jobs = []
    all_groups = sorted(group_to_machines.keys())
    for _, r in orders.iterrows():
        part_no = str(r['part_no'])
        q = int(r['quantity'])
        diam = float(part_diam.get(part_no, ops['diameter'].dropna().median()))

        eg10 = sorted(list(eligible[part_no].get(10, set(all_groups))))
        eg20 = sorted(list(eligible[part_no].get(20, set(eg10 if eg10 else all_groups))))
        if not eg10:
            eg10 = all_groups
        if not eg20:
            eg20 = eg10

        jobs.append(Job(
            job_id=r['job_id'],
            part_no=part_no,
            due_date=r['due_date'],
            quantity=q,
            diameter=diam,
            eligible_groups_op10=eg10,
            eligible_groups_op20=eg20,
        ))

    ops['shipment_file_overtime_min'] = ops.get('overtime_text', pd.Series(dtype=object)).apply(parse_overtime_text)
    baseline_overtime_min = float(ops['shipment_file_overtime_min'].sum())

    planning_start = datetime.combine(min([j.due_date.date() for j in jobs]), time(PLANNING_START_HOUR, 0))

    return {
        'ops': ops,
        'baseline_overtime_min': baseline_overtime_min,
        'orders': orders,
        'jobs': jobs,
        'group_to_machines': group_to_machines,
        'machine_to_group': machine_to_group,
        'setup_dict': setup_dict,
        'unit_time': unit_time,
        'fallback_part_op': fallback_part_op,
        'fallback_op': fallback_op,
        'global_unit': global_unit,
        'planning_start': planning_start,
    }


def get_unit_time(data, part_no, operation, group):
    key = (str(part_no), int(operation), int(group))
    if key in data['unit_time']:
        return data['unit_time'][key]
    key2 = (str(part_no), int(operation))
    if key2 in data['fallback_part_op']:
        return float(data['fallback_part_op'][key2])
    if int(operation) in data['fallback_op']:
        return float(data['fallback_op'][int(operation)])
    return data['global_unit']


def get_setup_time(data, from_diam, to_diam):
    if from_diam is None:
        return INITIAL_SETUP_MIN
    d1 = float(from_diam)
    d2 = float(to_diam)
    if abs(d1 - d2) < 1e-9:
        return SAME_DIAM_SETUP_MIN
    if (d1, d2) in data['setup_dict']:
        return data['setup_dict'][(d1, d2)]
    return DEFAULT_DIFF_DIAM_SETUP_MIN


def common_eligible_groups(job):
    common = sorted(list(set(job.eligible_groups_op10).intersection(set(job.eligible_groups_op20))))
    if common:
        return common
    return sorted(job.eligible_groups_op10)


# ============================================================
# 5. EXACT GUROBI MODEL
# ============================================================

def build_overtime_windows(planning_start, latest_due):
    horizon_end = latest_due + timedelta(minutes=TARDINESS_LIMIT_MIN) + timedelta(days=1)
    windows = []
    current_day = planning_start.date()
    while datetime.combine(current_day, time(0, 0)) <= horizon_end:
        w_start = datetime.combine(current_day, OVERTIME_START)
        w_end = datetime.combine(current_day, OVERTIME_END)
        u = dt_to_min(w_end, planning_start)
        l = dt_to_min(w_start, planning_start)
        if u > 0:
            windows.append((max(0.0, l), u, current_day.isoformat()))
        current_day = current_day + timedelta(days=1)
    return windows


def solve_exact_gurobi(data):
    jobs = data['jobs']

    # Safety check before creating Gurobi tupledict variables.
    # This catches duplicate job IDs with an explicit message instead of
    # Gurobi's less informative 'Duplicate keys in Model.addVars()' error.
    job_ids = [j.job_id for j in jobs]
    if len(job_ids) != len(set(job_ids)):
        vc = pd.Series(job_ids).value_counts()
        duplicates = vc[vc > 1].index.tolist()
        raise ValueError(f'Duplicate job_id values remained before model creation: {duplicates[:20]}')

    machines = sorted(data['machine_to_group'].keys())
    groups = sorted(data['group_to_machines'].keys())
    ops = [10, 20]
    planning_start = data['planning_start']
    latest_due = max(j.due_date for j in jobs)
    overtime_windows = build_overtime_windows(planning_start, latest_due)

    max_splits = MAX_SPLITS if ALLOW_JOB_SPLITTING else 1
    split_ids = list(range(1, max_splits + 1))

    # Candidate tasks are optional split-operation objects.
    tasks = []
    for j in jobs:
        for r in split_ids:
            for op in ops:
                tasks.append((j.job_id, r, op))

    job_by_id = {j.job_id: j for j in jobs}
    allowed_groups = {j.job_id: common_eligible_groups(j) for j in jobs}

    # Eligible machines for each task are determined by selected common group.
    eligible_machines_by_task = {}
    for j in jobs:
        for r in split_ids:
            for op in ops:
                ms = []
                for g in allowed_groups[j.job_id]:
                    ms.extend(data['group_to_machines'].get(g, []))
                eligible_machines_by_task[(j.job_id, r, op)] = sorted(set(ms))

    # Big-M horizon: tardiness limit already bounds final job completion.
    horizon_min = dt_to_min(latest_due + timedelta(minutes=TARDINESS_LIMIT_MIN) + timedelta(days=1), planning_start)
    M = max(BIG_M, horizon_min + sum(j.quantity for j in jobs) * 10.0)

    model = Model('Exact_Gurobi_Split_GlobalMachineNoOverlap')
    model.Params.MIPGap = MIP_GAP
    model.Params.OutputFlag = OUTPUT_FLAG
    if TIME_LIMIT_SEC is not None:
        model.Params.TimeLimit = TIME_LIMIT_SEC

    # --------------------------------------------------------
    # Variables
    # --------------------------------------------------------
    active = model.addVars(
        [(j.job_id, r) for j in jobs for r in split_ids],
        vtype=GRB.BINARY,
        name='active'
    )
    qty = model.addVars(
        [(j.job_id, r) for j in jobs for r in split_ids],
        vtype=GRB.INTEGER,
        lb=0,
        name='qty'
    )
    z_group = model.addVars(
        [(j.job_id, r, g) for j in jobs for r in split_ids for g in allowed_groups[j.job_id]],
        vtype=GRB.BINARY,
        name='z_group'
    )
    qty_group = model.addVars(
        [(j.job_id, r, g) for j in jobs for r in split_ids for g in allowed_groups[j.job_id]],
        vtype=GRB.INTEGER,
        lb=0,
        name='qty_group'
    )

    # x[t,m]
    x = model.addVars(
        [(t[0], t[1], t[2], m) for t in tasks for m in eligible_machines_by_task[t]],
        vtype=GRB.BINARY,
        name='x'
    )

    S = model.addVars(tasks, lb=0.0, name='S')
    C = model.addVars(tasks, lb=0.0, name='C')
    proc_time = model.addVars(tasks, lb=0.0, name='proc_time')

    # Immediate predecessor/successor variables on each physical machine.
    arc_keys = []
    for idx, t in enumerate(tasks):
        for u in tasks:
            if t == u:
                continue
            common_ms = set(eligible_machines_by_task[t]).intersection(eligible_machines_by_task[u])
            for m in common_ms:
                arc_keys.append((t[0], t[1], t[2], u[0], u[1], u[2], m))
    arc = model.addVars(arc_keys, vtype=GRB.BINARY, name='arc')

    first = model.addVars(
        [(t[0], t[1], t[2], m) for t in tasks for m in eligible_machines_by_task[t]],
        vtype=GRB.BINARY,
        name='first'
    )
    last = model.addVars(
        [(t[0], t[1], t[2], m) for t in tasks for m in eligible_machines_by_task[t]],
        vtype=GRB.BINARY,
        name='last'
    )

    job_finish = model.addVars([j.job_id for j in jobs], lb=0.0, name='job_finish')
    makespan = model.addVar(lb=0.0, name='makespan')

    # Overtime overlap variables for processing intervals only.
    ot_vars = {}
    for t in tasks:
        for w_idx, (_l, _u, _name) in enumerate(overtime_windows):
            ot_vars[(t, w_idx)] = model.addVar(lb=0.0, name=f'ot_{t[0]}_{t[1]}_{t[2]}_{w_idx}')

    # Optional binaries to disallow operation starts inside OT windows.
    start_side = {}
    if DISALLOW_START_IN_OVERTIME:
        for t in tasks:
            for w_idx, (_l, _u, _name) in enumerate(overtime_windows):
                start_side[(t, w_idx)] = model.addVar(vtype=GRB.BINARY, name=f'start_side_{t[0]}_{t[1]}_{t[2]}_{w_idx}')

    # --------------------------------------------------------
    # Split quantity and group constraints
    # --------------------------------------------------------
    for j in jobs:
        jid = j.job_id
        Q = int(j.quantity)
        min_q = min(MIN_SPLIT_QTY, Q)

        model.addConstr(active[jid, 1] == 1, name=f'first_split_active_{jid}')
        model.addConstr(quicksum(qty[jid, r] for r in split_ids) == Q, name=f'qty_conservation_{jid}')

        for r in split_ids:
            if r > 1:
                model.addConstr(active[jid, r] <= active[jid, r - 1], name=f'split_order_{jid}_{r}')
                if Q < 2 * MIN_SPLIT_QTY:
                    model.addConstr(active[jid, r] == 0, name=f'no_small_split_{jid}_{r}')

            model.addConstr(qty[jid, r] <= Q * active[jid, r], name=f'qty_ub_{jid}_{r}')
            model.addConstr(qty[jid, r] >= min_q * active[jid, r], name=f'qty_lb_{jid}_{r}')
            model.addConstr(
                quicksum(z_group[jid, r, g] for g in allowed_groups[jid]) == active[jid, r],
                name=f'group_select_{jid}_{r}'
            )
            model.addConstr(
                quicksum(qty_group[jid, r, g] for g in allowed_groups[jid]) == qty[jid, r],
                name=f'qty_group_sum_{jid}_{r}'
            )
            for g in allowed_groups[jid]:
                model.addConstr(qty_group[jid, r, g] <= Q * z_group[jid, r, g], name=f'qg_ub1_{jid}_{r}_{g}')
                model.addConstr(qty_group[jid, r, g] <= qty[jid, r], name=f'qg_ub2_{jid}_{r}_{g}')
                model.addConstr(qty_group[jid, r, g] >= qty[jid, r] - Q * (1 - z_group[jid, r, g]), name=f'qg_lb_{jid}_{r}_{g}')

    # --------------------------------------------------------
    # Assignment and same-group machine constraints
    # --------------------------------------------------------
    for t in tasks:
        jid, r, op = t
        model.addConstr(
            quicksum(x[jid, r, op, m] for m in eligible_machines_by_task[t]) == active[jid, r],
            name=f'assign_{jid}_{r}_{op}'
        )
        for m in eligible_machines_by_task[t]:
            g_m = data['machine_to_group'][m]
            if g_m in allowed_groups[jid]:
                model.addConstr(x[jid, r, op, m] <= z_group[jid, r, g_m], name=f'machine_group_link_{jid}_{r}_{op}_{m}')
            else:
                model.addConstr(x[jid, r, op, m] == 0, name=f'ineligible_machine_{jid}_{r}_{op}_{m}')

    # --------------------------------------------------------
    # Processing time and completion definitions
    # --------------------------------------------------------
    for t in tasks:
        jid, r, op = t
        job = job_by_id[jid]
        p_expr = quicksum(
            get_unit_time(data, job.part_no, op, g) * qty_group[jid, r, g]
            for g in allowed_groups[jid]
        )
        model.addConstr(proc_time[t] == p_expr, name=f'proc_time_def_{jid}_{r}_{op}')
        model.addConstr(C[t] == S[t] + proc_time[t], name=f'completion_{jid}_{r}_{op}')
        model.addConstr(S[t] <= M * active[jid, r], name=f'inactive_start_{jid}_{r}_{op}')
        model.addConstr(C[t] <= M * active[jid, r], name=f'inactive_finish_{jid}_{r}_{op}')

    # Technological precedence: Op20 after same split's Op10.
    for j in jobs:
        for r in split_ids:
            model.addConstr(
                S[j.job_id, r, 20] >= C[j.job_id, r, 10] - M * (1 - active[j.job_id, r]),
                name=f'tech_prec_{j.job_id}_{r}'
            )

    # Job-level finish and hard tardiness limit.
    for j in jobs:
        due_min = dt_to_min(j.due_date, planning_start)
        for r in split_ids:
            model.addConstr(job_finish[j.job_id] >= C[j.job_id, r, 20], name=f'job_finish_{j.job_id}_{r}')
        model.addConstr(
            job_finish[j.job_id] <= due_min + TARDINESS_LIMIT_MIN,
            name=f'tardiness_limit_{j.job_id}'
        )
        model.addConstr(makespan >= job_finish[j.job_id], name=f'makespan_{j.job_id}')

    # --------------------------------------------------------
    # Global machine sequence: Op10 and Op20 share same physical machine resource.
    # Immediate chain constraints: one predecessor/first and one successor/last per assigned task.
    # --------------------------------------------------------
    for m in machines:
        possible_tasks_m = [t for t in tasks if m in eligible_machines_by_task[t]]
        if not possible_tasks_m:
            continue

        model.addConstr(
            quicksum(first[t[0], t[1], t[2], m] for t in possible_tasks_m) <= 1,
            name=f'at_most_one_first_{m}'
        )
        model.addConstr(
            quicksum(last[t[0], t[1], t[2], m] for t in possible_tasks_m) <= 1,
            name=f'at_most_one_last_{m}'
        )

        for t in possible_tasks_m:
            tid = (t[0], t[1], t[2], m)
            pred_arcs = []
            succ_arcs = []
            for u in possible_tasks_m:
                if u == t:
                    continue
                key_pred = (u[0], u[1], u[2], t[0], t[1], t[2], m)
                key_succ = (t[0], t[1], t[2], u[0], u[1], u[2], m)
                if key_pred in arc:
                    pred_arcs.append(arc[key_pred])
                if key_succ in arc:
                    succ_arcs.append(arc[key_succ])

            model.addConstr(quicksum(pred_arcs) + first[tid] == x[tid], name=f'pred_or_first_{t}_{m}')
            model.addConstr(quicksum(succ_arcs) + last[tid] == x[tid], name=f'succ_or_last_{t}_{m}')
            model.addConstr(first[tid] <= x[tid], name=f'first_link_{t}_{m}')
            model.addConstr(last[tid] <= x[tid], name=f'last_link_{t}_{m}')
            model.addConstr(S[t] >= INITIAL_SETUP_MIN - M * (1 - first[tid]), name=f'initial_setup_time_{t}_{m}')

    for key in arc_keys:
        i_jid, i_r, i_op, j_jid, j_r, j_op, m = key
        ti = (i_jid, i_r, i_op)
        tj = (j_jid, j_r, j_op)
        xi = x[i_jid, i_r, i_op, m]
        xj = x[j_jid, j_r, j_op, m]
        model.addConstr(arc[key] <= xi, name=f'arc_link_i_{key}')
        model.addConstr(arc[key] <= xj, name=f'arc_link_j_{key}')
        setup_ij = get_setup_time(data, job_by_id[i_jid].diameter, job_by_id[j_jid].diameter)
        model.addConstr(
            S[tj] >= C[ti] + setup_ij - M * (1 - arc[key]),
            name=f'global_no_overlap_setup_{key}'
        )

    # --------------------------------------------------------
    # Overtime overlap via Gurobi general constraints.
    # ov = max(0, min(C, window_end) - max(S, window_start))
    # --------------------------------------------------------
    for t in tasks:
        jid, r, op = t
        for w_idx, (l, u, _wname) in enumerate(overtime_windows):
            max_start = model.addVar(lb=-M, ub=M, name=f'max_start_{jid}_{r}_{op}_{w_idx}')
            min_finish = model.addVar(lb=-M, ub=M, name=f'min_finish_{jid}_{r}_{op}_{w_idx}')
            diff = model.addVar(lb=-M, ub=M, name=f'overlap_diff_{jid}_{r}_{op}_{w_idx}')
            model.addGenConstrMax(max_start, [S[t]], constant=float(l), name=f'gc_max_start_{jid}_{r}_{op}_{w_idx}')
            model.addGenConstrMin(min_finish, [C[t]], constant=float(u), name=f'gc_min_finish_{jid}_{r}_{op}_{w_idx}')
            model.addConstr(diff == min_finish - max_start, name=f'overlap_diff_def_{jid}_{r}_{op}_{w_idx}')
            model.addGenConstrMax(ot_vars[(t, w_idx)], [diff], constant=0.0, name=f'gc_overlap_pos_{jid}_{r}_{op}_{w_idx}')
            model.addConstr(ot_vars[(t, w_idx)] <= (u - l) * active[jid, r], name=f'ot_active_ub_{jid}_{r}_{op}_{w_idx}')

            if DISALLOW_START_IN_OVERTIME:
                b = start_side[(t, w_idx)]
                model.addConstr(S[t] <= float(l) + M * b + M * (1 - active[jid, r]), name=f'start_before_or_after_1_{jid}_{r}_{op}_{w_idx}')
                model.addConstr(S[t] >= float(u) - M * (1 - b) - M * (1 - active[jid, r]), name=f'start_before_or_after_2_{jid}_{r}_{op}_{w_idx}')

    total_overtime = quicksum(ot_vars[(t, w_idx)] for t in tasks for w_idx in range(len(overtime_windows)))
    total_setup_surrogate = (
        quicksum(INITIAL_SETUP_MIN * first[key] for key in first.keys())
        + quicksum(get_setup_time(data, job_by_id[k[0]].diameter, job_by_id[k[3]].diameter) * arc[k] for k in arc_keys)
    )

    model.setObjective(
        W_OVERTIME * total_overtime
        + W_MAKESPAN * makespan
        + W_SETUP * total_setup_surrogate,
        GRB.MINIMIZE
    )

    model.update()
    print('\nModel size:')
    print('Variables   :', model.NumVars)
    print('Constraints :', model.NumConstrs)
    print('GenConstrs  :', model.NumGenConstrs)

    model.optimize()

    if model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        if model.Status == GRB.INFEASIBLE:
            print('Model infeasible. Computing IIS...')
            model.computeIIS()
            iis_path = BASE_DIR / 'model_iis.ilp'
            model.write(str(iis_path))
            print('IIS written to:', iis_path)
        raise RuntimeError(f'Gurobi did not return a usable solution. Status={model.Status}')

    sol = {
        'model': model,
        'active': active,
        'qty': qty,
        'z_group': z_group,
        'qty_group': qty_group,
        'x': x,
        'S': S,
        'C': C,
        'proc_time': proc_time,
        'arc': arc,
        'first': first,
        'last': last,
        'job_finish': job_finish,
        'makespan': makespan,
        'ot_vars': ot_vars,
        'overtime_windows': overtime_windows,
        'tasks': tasks,
        'machines': machines,
        'split_ids': split_ids,
        'allowed_groups': allowed_groups,
        'eligible_machines_by_task': eligible_machines_by_task,
        'job_by_id': job_by_id,
        'objective_total_overtime': total_overtime.getValue(),
        'objective_setup_surrogate': total_setup_surrogate.getValue(),
        'objective_makespan': makespan.X,
    }
    return sol


# ============================================================
# 6. SOLUTION EXTRACTION, VERIFICATION, OUTPUTS
# ============================================================

def value(v):
    try:
        return float(v.X)
    except Exception:
        return float(v)


def extract_solution_tables(data, sol):
    model = sol['model']
    jobs = data['jobs']
    job_by_id = sol['job_by_id']
    machines = sol['machines']
    split_ids = sol['split_ids']
    tasks = sol['tasks']
    active = sol['active']
    qty = sol['qty']
    z_group = sol['z_group']
    x = sol['x']
    S = sol['S']
    C = sol['C']
    proc_time = sol['proc_time']
    arc = sol['arc']
    first = sol['first']
    last = sol['last']
    planning_start = data['planning_start']

    # active split data
    split_rows = []
    selected_qty = {}
    selected_group = {}
    for j in jobs:
        for r in split_ids:
            a = value(active[j.job_id, r])
            q = round(value(qty[j.job_id, r]))
            if a > 0.5:
                g_sel = None
                for g in sol['allowed_groups'][j.job_id]:
                    if value(z_group[j.job_id, r, g]) > 0.5:
                        g_sel = g
                        break
                selected_qty[(j.job_id, r)] = int(q)
                selected_group[(j.job_id, r)] = g_sel
                split_rows.append({
                    'job_id': j.job_id,
                    'part_no': j.part_no,
                    'split_id': r,
                    'quantity': int(q),
                    'selected_group': g_sel,
                    'original_quantity': j.quantity,
                    'due_date': j.due_date,
                })

    # selected machine per task
    selected_machine = {}
    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5:
            continue
        for m in sol['eligible_machines_by_task'][t]:
            if (jid, r, op, m) in x and value(x[jid, r, op, m]) > 0.5:
                selected_machine[t] = m
                break

    # direct predecessor based on immediate arcs
    pred_task = {}
    setup_by_task = {}
    first_flag = {}
    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5:
            continue
        m = selected_machine[t]
        first_flag[t] = value(first[jid, r, op, m]) > 0.5
        if first_flag[t]:
            pred_task[t] = None
            setup_by_task[t] = INITIAL_SETUP_MIN
        else:
            pred = None
            setup = 0.0
            for key, var in arc.items():
                i_jid, i_r, i_op, j_jid, j_r, j_op, mm = key
                if mm == m and (j_jid, j_r, j_op) == t and value(var) > 0.5:
                    pred = (i_jid, i_r, i_op)
                    setup = get_setup_time(data, job_by_id[i_jid].diameter, job_by_id[j_jid].diameter)
                    break
            pred_task[t] = pred
            setup_by_task[t] = setup

    # schedule rows sorted by actual processing start on each physical machine
    rows = []
    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5:
            continue
        job = job_by_id[jid]
        m = selected_machine[t]
        start_min = value(S[t])
        finish_min = value(C[t])
        process_min = value(proc_time[t])
        setup_min = setup_by_task[t]
        setup_start_min = max(0.0, start_min - setup_min)
        pred = pred_task[t]
        overtime_min = overtime_overlap_minutes(min_to_dt(start_min, planning_start), min_to_dt(finish_min, planning_start))

        rows.append({
            'job_id': jid,
            'part_no': job.part_no,
            'split_id': r,
            'split_count': sum(1 for rr in split_ids if value(active[jid, rr]) > 0.5),
            'quantity': selected_qty[(jid, r)],
            'diameter': job.diameter,
            'group': selected_group[(jid, r)],
            'operation': op,
            'machine': m,
            'start_min': start_min,
            'finish_min': finish_min,
            'setup_start_min': setup_start_min,
            'setup_finish_min': start_min,
            'processing_min': process_min,
            'setup_min': setup_min,
            'overtime_min': overtime_min,
            'start': min_to_dt(start_min, planning_start),
            'finish': min_to_dt(finish_min, planning_start),
            'setup_start': min_to_dt(setup_start_min, planning_start),
            'setup_finish': min_to_dt(start_min, planning_start),
            'prev_task': 'SRC' if pred is None else f'{job_by_id[pred[0]].part_no} S{pred[1]} Op{pred[2]}',
            'is_first_on_machine': pred is None,
        })

    schedule_df = pd.DataFrame(rows)
    if not schedule_df.empty:
        schedule_df = schedule_df.sort_values(['machine', 'start_min', 'finish_min']).reset_index(drop=True)
        schedule_df['global_sequence_on_machine'] = schedule_df.groupby('machine').cumcount() + 1

    return schedule_df, pd.DataFrame(split_rows)


def check_machine_global_nooverlap(schedule_df, tolerance_min=1e-5):
    rows = []
    df = schedule_df.copy().sort_values(['machine', 'start_min', 'finish_min'])
    for machine, g in df.groupby('machine'):
        recs = g.to_dict('records')
        for i in range(len(recs) - 1):
            cur = recs[i]
            nxt = recs[i + 1]
            required_start = cur['finish_min'] + get_setup_time(GLOBAL_DATA, cur['diameter'], nxt['diameter'])
            slack = nxt['start_min'] - required_start
            overlap_proc = cur['finish_min'] - nxt['start_min']
            rows.append({
                'machine': machine,
                'current_task': f"{cur['part_no']} S{cur['split_id']} Op{cur['operation']}",
                'next_task': f"{nxt['part_no']} S{nxt['split_id']} Op{nxt['operation']}",
                'current_finish_min': cur['finish_min'],
                'required_next_start_after_setup': required_start,
                'next_start_min': nxt['start_min'],
                'setup_between_min': get_setup_time(GLOBAL_DATA, cur['diameter'], nxt['diameter']),
                'slack_min': slack,
                'processing_overlap_min': max(0.0, overlap_proc),
                'Feasible': slack >= -tolerance_min,
            })
    if not rows:
        return pd.DataFrame([{'status': 'NO PAIRS TO CHECK', 'Feasible': True}])
    return pd.DataFrame(rows)


def build_validation_tables(data, sol, schedule_df, split_df):
    jobs = data['jobs']
    split_ids = sol['split_ids']
    tasks = sol['tasks']
    active = sol['active']
    qty = sol['qty']
    S = sol['S']
    C = sol['C']
    proc_time = sol['proc_time']
    x = sol['x']
    job_by_id = sol['job_by_id']

    assignment_rows = []
    completion_rows = []
    precedence_rows = []
    tardiness_rows = []
    quantity_rows = []
    machine_nooverlap_df = check_machine_global_nooverlap(schedule_df)

    for j in jobs:
        total_q = sum(round(value(qty[j.job_id, r])) for r in split_ids)
        quantity_rows.append({
            'job_id': j.job_id,
            'part_no': j.part_no,
            'original_quantity': j.quantity,
            'sum_split_quantity': total_q,
            'Feasible': abs(total_q - j.quantity) <= EPS,
        })
        finish_vals = []
        for r in split_ids:
            if value(active[j.job_id, r]) > 0.5:
                finish_vals.append(value(C[j.job_id, r, 20]))
        job_finish = max(finish_vals) if finish_vals else 0.0
        due_min = dt_to_min(j.due_date, data['planning_start'])
        tard = max(0.0, job_finish - due_min)
        viol = max(0.0, job_finish - (due_min + TARDINESS_LIMIT_MIN))
        tardiness_rows.append({
            'job_id': j.job_id,
            'part_no': j.part_no,
            'job_finish_min': job_finish,
            'due_min': due_min,
            'tardiness_min_from_due': tard,
            'tardiness_limit_min': TARDINESS_LIMIT_MIN,
            'tardiness_limit_violation_min': viol,
            'Feasible': viol <= EPS,
        })

    for t in tasks:
        jid, r, op = t
        a = value(active[jid, r])
        assign_sum = sum(value(x[jid, r, op, m]) for m in sol['eligible_machines_by_task'][t] if (jid, r, op, m) in x)
        assignment_rows.append({
            'job_id': jid,
            'split_id': r,
            'operation': op,
            'active': a,
            'assignment_sum': assign_sum,
            'Feasible': abs(assign_sum - a) <= EPS,
        })
        lhs = value(C[t])
        rhs = value(S[t]) + value(proc_time[t])
        completion_rows.append({
            'job_id': jid,
            'split_id': r,
            'operation': op,
            'C': lhs,
            'S_plus_processing': rhs,
            'Deviation': abs(lhs - rhs),
            'Feasible': abs(lhs - rhs) <= EPS,
        })

    for j in jobs:
        for r in split_ids:
            if value(active[j.job_id, r]) > 0.5:
                slack = value(S[j.job_id, r, 20]) - value(C[j.job_id, r, 10])
                precedence_rows.append({
                    'job_id': j.job_id,
                    'split_id': r,
                    'start_op20_min': value(S[j.job_id, r, 20]),
                    'finish_op10_min': value(C[j.job_id, r, 10]),
                    'slack_min': slack,
                    'Feasible': slack >= -EPS,
                })

    dfs = {
        'Split_Quantity_Check': pd.DataFrame(quantity_rows),
        'Assignment_Check': pd.DataFrame(assignment_rows),
        'Completion_Check': pd.DataFrame(completion_rows),
        'Precedence_Check': pd.DataFrame(precedence_rows),
        'Tardiness_Check': pd.DataFrame(tardiness_rows),
        'Machine_Global_NoOverlap': machine_nooverlap_df,
    }

    summary = []
    for name, df in dfs.items():
        if 'Feasible' in df.columns:
            summary.append({
                'Check': name,
                'All_Feasible': bool(df['Feasible'].all()),
                'Violations': int((~df['Feasible']).sum()),
            })
    dfs['Feasibility_Summary'] = pd.DataFrame(summary)
    return dfs


def build_machine_summary(schedule_df):
    df = schedule_df.copy()
    return df.groupby('machine').agg(
        total_processing_min=('processing_min', 'sum'),
        total_setup_min=('setup_min', 'sum'),
        total_overtime_min=('overtime_min', 'sum'),
        first_start=('start', 'min'),
        last_finish=('finish', 'max'),
        task_count=('job_id', 'count'),
    ).reset_index().sort_values('total_overtime_min', ascending=False)


def build_job_summary(schedule_df, data):
    jobs_by_id = {j.job_id: j for j in data['jobs']}
    comp = schedule_df[schedule_df['operation'] == 20].groupby('job_id')['finish'].max().reset_index()
    rows = []
    for _, r in comp.iterrows():
        job = jobs_by_id[r['job_id']]
        finish = r['finish']
        tardiness = max(0.0, (finish - job.due_date).total_seconds() / 60.0)
        rows.append({
            'job_id': job.job_id,
            'part_no': job.part_no,
            'quantity': job.quantity,
            'due_date': job.due_date,
            'completion': finish,
            'tardiness_min_from_due': tardiness,
            'tardiness_days_from_due': tardiness / 1440.0,
            'allowed_latest_completion': job.due_date + timedelta(minutes=TARDINESS_LIMIT_MIN),
            'tardiness_limit_violation_min': max(0.0, (finish - (job.due_date + timedelta(minutes=TARDINESS_LIMIT_MIN))).total_seconds() / 60.0),
        })
    return pd.DataFrame(rows).sort_values(['due_date', 'part_no'])


def build_split_summary(schedule_df, data):
    rows = []
    jobs_by_id = {j.job_id: j for j in data['jobs']}
    for job_id, g in schedule_df.groupby('job_id'):
        job = jobs_by_id[job_id]
        split_quantities = (
            g[['split_id', 'quantity']]
            .drop_duplicates()
            .sort_values('split_id')
            .apply(lambda r: f"S{int(r['split_id'])}={int(r['quantity'])}", axis=1)
            .tolist()
        )
        rows.append({
            'job_id': job_id,
            'part_no': job.part_no,
            'original_quantity': job.quantity,
            'split_count': int(g['split_count'].max()),
            'is_split': 'YES' if int(g['split_count'].max()) > 1 else 'NO',
            'split_quantities': ', '.join(split_quantities),
            'due_date': job.due_date,
            'final_completion': g[g['operation'] == 20]['finish'].max(),
        })
    return pd.DataFrame(rows).sort_values(['is_split', 'due_date', 'part_no'], ascending=[False, True, True])


def export_excel_outputs(data, sol, schedule_df, split_df, validation_dfs):
    machine_summary = build_machine_summary(schedule_df)
    job_summary = build_job_summary(schedule_df, data)
    split_summary = build_split_summary(schedule_df, data)

    metrics = pd.DataFrame([{
        'objective_value': sol['model'].ObjVal,
        'total_processing_overtime_min': schedule_df['overtime_min'].sum(),
        'baseline_overtime_min': data.get('baseline_overtime_min', 0.0),
        'overtime_improvement_min': data.get('baseline_overtime_min', 0.0) - schedule_df['overtime_min'].sum(),
        'total_setup_min': schedule_df['setup_min'].sum(),
        'makespan_min': sol['objective_makespan'],
        'gurobi_status': sol['model'].Status,
        'mip_gap': sol['model'].MIPGap if hasattr(sol['model'], 'MIPGap') else None,
    }])

    schedule_out = schedule_df.copy()
    schedule_out['Tarih'] = pd.to_datetime(schedule_out['start']).dt.date
    schedule_out['Başlangıç Saat'] = pd.to_datetime(schedule_out['start']).dt.strftime('%H:%M')
    schedule_out['Bitiş Saat'] = pd.to_datetime(schedule_out['finish']).dt.strftime('%H:%M')
    schedule_out['Parça No'] = schedule_out['part_no']
    schedule_out['Makine No'] = schedule_out['machine']
    schedule_out['CNC-1 operasyonu(piston)'] = np.where(schedule_out['operation'] == 10, 'X', '')
    schedule_out['CNC-2 operasyonu (saplama)'] = np.where(schedule_out['operation'] == 20, 'X', '')
    schedule_out['Adet'] = schedule_out['quantity']
    schedule_out['Çap'] = schedule_out['diameter']
    schedule_out['Makine Grubu'] = schedule_out['group']
    schedule_out['Fazla mesai'] = schedule_out['overtime_min'].round(1)
    schedule_out['Split'] = schedule_out['split_id']
    schedule_out['Split Count'] = schedule_out['split_count']
    schedule_out['Split Label'] = 'S' + schedule_out['split_id'].astype(str) + '/' + schedule_out['split_count'].astype(str)
    schedule_out['Operation Label'] = np.where(schedule_out['operation'] == 10, 'Op10', 'Op20')
    schedule_out['Setup Süresi'] = schedule_out['setup_min'].round(1)
    schedule_out['İşlem Süresi'] = schedule_out['processing_min'].round(1)
    schedule_out['Job ID'] = schedule_out['job_id']

    with pd.ExcelWriter(OUTPUT_SCHEDULE_FILE, engine='openpyxl') as writer:
        schedule_out.to_excel(writer, sheet_name='Optimized Schedule', index=False)
        machine_summary.to_excel(writer, sheet_name='Machine Summary', index=False)
        job_summary.to_excel(writer, sheet_name='Job Summary', index=False)
        split_summary.to_excel(writer, sheet_name='Split Summary', index=False)
        metrics.to_excel(writer, sheet_name='Final Metrics', index=False)

    with pd.ExcelWriter(OUTPUT_VALIDATION_FILE, engine='openpyxl') as writer:
        validation_dfs['Feasibility_Summary'].to_excel(writer, sheet_name='Feasibility_Summary', index=False)
        for name, df in validation_dfs.items():
            if name == 'Feasibility_Summary':
                continue
            df.to_excel(writer, sheet_name=name[:31], index=False)


def export_gantt_html(schedule_df, output_html_path):
    if schedule_df.empty:
        with open(output_html_path, 'w', encoding='utf-8') as f:
            f.write('<html><body><h2>No schedule data.</h2></body></html>')
        return

    records = []
    palette = {
        'Op.10': '#3b82f6',
        'Op.10 Split': '#f59e0b',
        'Op.20': '#10b981',
        'Op.20 Split': '#f97316',
        'Setup': '#111111',
    }

    for _, task in schedule_df.iterrows():
        split_flag = int(task['split_count']) > 1
        legend = 'Op.10 Split' if (task['operation'] == 10 and split_flag) else \
                 'Op.10' if task['operation'] == 10 else \
                 'Op.20 Split' if split_flag else 'Op.20'
        label = f"{task['part_no']} S{int(task['split_id'])}/{int(task['split_count'])} | Op{int(task['operation'])}"

        if task['setup_min'] > EPS:
            records.append({
                'Legend': 'Setup',
                'Machine': task['machine'],
                'Start_min': task['setup_start_min'],
                'Finish_min': task['setup_finish_min'],
                'Duration_min': task['setup_min'],
                'Text': '',
                'Task': f"SETUP -> {label}",
                'Job': task['job_id'],
                'Part': task['part_no'],
                'Split': task['split_id'],
                'Operation': task['operation'],
                'Qty': task['quantity'],
                'Setup_min': task['setup_min'],
                'Processing_min': 0,
                'Start_time': str(task['setup_start']),
                'Finish_time': str(task['setup_finish']),
            })

        records.append({
            'Legend': legend,
            'Machine': task['machine'],
            'Start_min': task['start_min'],
            'Finish_min': task['finish_min'],
            'Duration_min': task['processing_min'],
            'Text': label,
            'Task': label,
            'Job': task['job_id'],
            'Part': task['part_no'],
            'Split': task['split_id'],
            'Operation': task['operation'],
            'Qty': task['quantity'],
            'Setup_min': task['setup_min'],
            'Processing_min': task['processing_min'],
            'Start_time': str(task['start']),
            'Finish_time': str(task['finish']),
        })

    gantt_df = pd.DataFrame(records).sort_values(['Machine', 'Start_min', 'Finish_min']).reset_index(drop=True)
    machine_order = sorted(gantt_df['Machine'].unique().tolist())
    legend_order = ['Op.10', 'Op.10 Split', 'Op.20', 'Op.20 Split', 'Setup']
    makespan = schedule_df['finish_min'].max()

    fig = go.Figure()
    for legend_name in legend_order:
        sub = gantt_df[gantt_df['Legend'] == legend_name].copy()
        if sub.empty:
            continue
        fig.add_trace(go.Bar(
            x=sub['Duration_min'].tolist(),
            y=sub['Machine'].tolist(),
            base=sub['Start_min'].tolist(),
            orientation='h',
            name=legend_name,
            marker=dict(
                color=palette[legend_name],
                line=dict(color='#111111' if 'Split' in legend_name else '#ffffff', width=1.4 if 'Split' in legend_name else 0.4),
                pattern=dict(shape='/' if 'Split' in legend_name else ''),
            ),
            text=sub['Text'].tolist(),
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(color='white', size=10),
            customdata=list(zip(
                sub['Job'], sub['Part'], sub['Split'], sub['Operation'], sub['Qty'],
                sub['Setup_min'], sub['Processing_min'], sub['Start_min'], sub['Finish_min'],
                sub['Start_time'], sub['Finish_time'], sub['Task']
            )),
            hovertemplate=(
                'Makine: %{y}<br>'
                'Task: %{customdata[11]}<br>'
                'Job: %{customdata[0]}<br>'
                'Part: %{customdata[1]}<br>'
                'Split: %{customdata[2]}<br>'
                'Operation: %{customdata[3]}<br>'
                'Qty: %{customdata[4]}<br>'
                'Setup min: %{customdata[5]:.2f}<br>'
                'Processing min: %{customdata[6]:.2f}<br>'
                'Start min: %{customdata[7]:.2f}<br>'
                'Finish min: %{customdata[8]:.2f}<br>'
                'Start time: %{customdata[9]}<br>'
                'Finish time: %{customdata[10]}<extra></extra>'
            )
        ))

    fig.add_shape(
        type='line',
        x0=makespan,
        x1=makespan,
        y0=0,
        y1=1,
        xref='x',
        yref='paper',
        line=dict(width=2, dash='dash', color='#dc2626'),
    )
    fig.add_annotation(
        x=makespan,
        y=1.02,
        xref='x',
        yref='paper',
        text=f'Cmax={makespan:.0f} dk',
        showarrow=False,
        xanchor='left',
        font=dict(color='#dc2626', size=11),
        bgcolor='rgba(255,255,255,0.7)',
    )

    fig.update_layout(
        barmode='overlay',
        title='Exact Gurobi Schedule Gantt Chart<br><sup>Setup=siyah, Op10=mavi, Op20=yeşil, split=desenli</sup>',
        xaxis_title='Zaman (dk)',
        yaxis_title='Makine',
        yaxis=dict(categoryorder='array', categoryarray=machine_order, autorange='reversed'),
        legend_title='Görev Türü',
        plot_bgcolor='white',
        paper_bgcolor='white',
        hovermode='closest',
        width=2500,
        height=max(650, 70 * len(machine_order)),
        margin=dict(l=80, r=40, t=110, b=60),
    )
    fig.update_xaxes(showgrid=True, gridcolor='#e5e7eb', zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor='#f3f4f6', zeroline=False)

    n = len(fig.data)
    def vis(names):
        return [fig.data[i].name in names for i in range(n)]
    fig.update_layout(updatemenus=[dict(
        type='buttons', direction='right', x=0, y=1.18, showactive=True,
        buttons=[
            dict(label='Tümü', method='update', args=[{'visible': [True] * n}]),
            dict(label='Sadece Operasyonlar', method='update', args=[{'visible': vis({'Op.10','Op.10 Split','Op.20','Op.20 Split'})}]),
            dict(label='Sadece Op.10', method='update', args=[{'visible': vis({'Op.10','Op.10 Split'})}]),
            dict(label='Sadece Op.20', method='update', args=[{'visible': vis({'Op.20','Op.20 Split'})}]),
            dict(label='Sadece Setup', method='update', args=[{'visible': vis({'Setup'})}]),
            dict(label='Setup Hariç', method='update', args=[{'visible': vis({'Op.10','Op.10 Split','Op.20','Op.20 Split'})}]),
        ],
    )])

    fig.write_html(str(output_html_path), include_plotlyjs='cdn', full_html=True)


# ============================================================
# 7. MAIN RUN
# ============================================================

if __name__ == '__main__':
    print('\nLoading data...')
    GLOBAL_DATA = build_problem_data(
        shipment_file=SHIPMENT_FILE,
        sdst_file=SDST_FILE,
        machine_group_file=MACHINE_GROUP_FILE,
        use_shipped_quantity=True,
    )

    print(f"Number of jobs: {len(GLOBAL_DATA['jobs'])}")
    print(f"Machines: {sorted(GLOBAL_DATA['machine_to_group'].keys())}")
    print(f"Planning start: {GLOBAL_DATA['planning_start']}")
    print(f"Tardiness limit days: {TARDINESS_LIMIT_DAYS}")

    print('\nBuilding and solving exact Gurobi MILP...')
    solution = solve_exact_gurobi(GLOBAL_DATA)

    print('\nExtracting solution...')
    schedule_df, split_df = extract_solution_tables(GLOBAL_DATA, solution)
    validation_dfs = build_validation_tables(GLOBAL_DATA, solution, schedule_df, split_df)

    print('Creating Excel outputs...')
    export_excel_outputs(GLOBAL_DATA, solution, schedule_df, split_df, validation_dfs)

    print('Creating HTML Gantt...')
    export_gantt_html(schedule_df, OUTPUT_GANTT_FILE)

    print('\nDone.')
    print('Schedule Excel   :', OUTPUT_SCHEDULE_FILE)
    print('Validation Excel :', OUTPUT_VALIDATION_FILE)
    print('Gantt HTML       :', OUTPUT_GANTT_FILE)
    print('Objective value  :', solution['model'].ObjVal)
    print('Total OT (min)   :', schedule_df['overtime_min'].sum())
import os
import psutil

process = psutil.Process(os.getpid())
peak_mb = process.memory_info().rss / 1024 / 1024
print(f"Peak Memory: {peak_mb:.2f} MB")

Excel files in /content:

Using files:
SHIPMENT_FILE      = C:\Users\emrec\Documents\parallel_scheduling\exact1\492-güncel sevkiyat.xlsx
SDST_FILE          = C:\Users\emrec\Documents\parallel_scheduling\exact1\SDST.xlsx
MACHINE_GROUP_FILE = C:\Users\emrec\Documents\parallel_scheduling\exact1\machine_group_data.xlsx
OUTPUT_SCHEDULE    = C:\Users\emrec\Documents\parallel_scheduling\exact1\optimized_exact_gurobi_schedule_split.xlsx
OUTPUT_VALIDATION  = C:\Users\emrec\Documents\parallel_scheduling\exact1\verification_validation_exact_gurobi_split.xlsx
OUTPUT_GANTT_HTML  = C:\Users\emrec\Documents\parallel_scheduling\exact1\exact_gurobi_gantt_split.html

Loading data...
Duplicate order rows aggregated: 28 rows -> 27 unique jobs
Number of jobs: 27
Machines: ['T.3', 'T.3.1', 'T.3.10', 'T.3.11', 'T.3.12', 'T.3.2', 'T.3.3', 'T.3.4', 'T.3.6', 'T.3.7', 'T.3.8', 'T.3.9']
Planning start: 2026-04-29 07:00:00
Tardiness limit days: 1.5

Building and solving exact Gurobi MILP...
Set parameter Username


  2106  1939  250.72028   53  160          -    8.95883      -   241  189s
  2107  1940  250.72028  102  212          -    8.95883      -   241  190s
  2108  1941   10.44248   55  166          -    8.96272      -   241  197s
  2110  1942   11.70283  223  169          -    8.96272      -   241  204s
  2111  1943   14.24888  368  175          -    8.96272      -   241  205s
  2112  1943   10.44248   48  150          -    8.96272      -   241  211s
  2114  1945  253.94888  283  196          -    8.96272      -   241  218s
  2115  1945   10.44248   55  263          -    8.96272      -   240  220s
  2117  1947   11.39641  135  336          -    8.96272      -   240  226s
  2119  1948    9.62402   73  175          -    8.96272      -   240  232s
  2120  1949  253.69694  386  175          -    8.96272      -   240  295s
  2123  1956   10.52028   29  243          -    9.20495      -   207  303s
  2131  1966   10.52028   31  249          -    9.30037      -   209  309s
  2140  1976   35.41800  

 29613 25876 infeasible  472               -    9.40391      -   113 1200s
 29786 26203   14.70849  493  161          -    9.40391      -   113 1206s
 30173 26775  205.11669   82  305          -    9.40391      -   113 1213s
 30796 26938  205.16011  145  250          -    9.40391      -   112 1218s
 30959 27344  205.25823  146  252          -    9.40391      -   112 1223s
 31367 28121  205.54439  237  249          -    9.40391      -   112 1250s
 32146 28701  205.54439  400  225          -    9.40391      -   111 1258s
 32804 29101  205.54439  475  153          -    9.47890      -   110 1263s
 33257 29251  209.16189  585  174          -    9.47890      -   110 1267s
 33413 29735  215.71223  592  191          -    9.47890      -   110 1272s
 33906 29999  221.81597  670  162          -    9.47890      -   109 1276s
 34171 30511 infeasible  693               -    9.47890      -   109 1281s
 34728 30948   11.94106   54  305          -    9.47890      -   109 1286s
 35178 31186   12.08800  

 72824 66827   12.34045  423  261          -    9.51841      -  94.6 1927s
 73275 67568   12.39045  566  223          -    9.51841      -  94.3 1954s
 74072 68001   12.54045  663  195          -    9.51841      -  94.1 1959s
 74542 68090   84.76921   44  309          -    9.51841      -  93.9 1962s
 74631 68328   92.21576   66  261          -    9.51841      -  94.0 1966s
 74874 68541   92.57373  125  194          -    9.51841      -  94.0 1970s
 75090 69054   92.57373  154  219          -    9.51841      -  94.0 1975s
 75613 69531   93.11379  374  173          -    9.51841      -  93.8 1981s
 76217 69767   12.18800   76  256          -    9.51841      -  93.4 1985s
 76464 70033   12.23800  155  212          -    9.51841      -  93.4 1990s
 77092 70771   12.92423  307  172          -    9.51841      -  93.3 1999s
 77538 71260   12.92423  373  173          -    9.51841      -  93.2 2004s
 78120 71541   12.92423  441  197          -    9.51841      -  92.9 2008s
 78403 71958   12.92699  

 117178 108007   14.81015   71  234          -    9.54594      -  88.7 2616s
 117542 108569   14.86015  144  220          -    9.54594      -  88.6 2622s
 118162 108699   98.98188  261  198          -    9.54594      -  88.6 2626s
 118296 109224   98.98188  273  205          -    9.54594      -  88.6 2631s
 118920 109312  105.76227  312  223          -    9.54594      -  88.5 2635s
 119052 109452   99.38188  366  188          -    9.54594      -  88.6 2640s
 119600 110076 infeasible  467               -    9.54594      -  88.6 2649s
 119885 110396   11.19438   41  272          -    9.54594      -  88.6 2653s
 120212 110686   11.86126   46  309          -    9.54594      -  88.6 2658s
 120504 111234   12.19576   82  210          -    9.54594      -  88.7 2664s
 121213 111374   12.29576  166  228          -    9.54594      -  88.4 2668s
 121353 111650   12.29576  219  244          -    9.54594      -  88.5 2672s
 121629 112082   12.29576  309  205          -    9.54594      -  88.5 2677s

...

 575101 530157  341.34696  276  192          -    9.68818      -  89.6 15281s
 575768 530793   12.37088  101  225          -    9.68821      -  89.7 15299s
 575905 530793   77.62589  100  220          -    9.68821      -  89.7 15300s
 576404 531594   13.53608  214  222          -    9.68821      -  89.7 15323s
 577301 531815  264.83237  328  193          -    9.68821      -  89.6 15333s
 577534 532156  269.07919  389  175          -    9.68821      -  89.7 15345s
 577929 532882   19.76241   98  227          -    9.68821      -  89.7 15366s
 578741 533115   19.86241  201  230          -    9.68821      -  89.6 15377s
 579030 533422   19.96241  240  241          -    9.68821      -  89.6 15389s
 579392 533763   20.07841  372  248          -    9.68821      -  89.6 15400s
 579733 534160   35.57130  407  232          -    9.68821      -  89.6 15413s
 580136 534664   20.17841  468  189          -    9.68821      -  89.6 15428s
 580704 535037   12.07525   39  341          -    9.68821      -

 614814 566589  178.03351  166  198          -    9.69214      -  89.9 16773s
 615349 567172  178.03351  272  197          -    9.69214      -  89.9 16792s
 615967 567814  178.09903  361  204          -    9.69214      -  89.9 16835s
 616742 567989   11.61306   58  276          -    9.69214      -  89.8 16843s
 616919 568426   12.17018   68  282          -    9.69214      -  89.8 16857s
 617416 568653   12.58800  135  156          -    9.69214      -  89.8 16866s
 617644 568878   12.58800  199  175          -    9.69214      -  89.9 16875s
 617869 569308   12.68800  255  183          -    9.69214      -  89.9 16890s
 618359 569653   12.68800  356  169          -    9.69214      -  89.8 16901s
 618705 569939   12.97890  390  168          -    9.69214      -  89.9 16913s
 618991 570370   14.17015  419  173          -    9.69214      -  89.9 16931s
 619578 570790    9.82391   55  337          -    9.69214      -  89.8 16945s
 619999 571150   68.97825   86  251          -    9.69214      -

 654644 603660   99.58567  490  262          -    9.69313      -  90.1 18439s
 654838 603660   12.85641  105  296          -    9.69313      -  90.1 18440s
 655115 604268  104.49246  532  261          -    9.69313      -  90.1 18459s
 655723 604841  104.69291  640  234          -    9.69313      -  90.0 18478s
 656298 605407  104.89384  731  231          -    9.69313      -  90.0 18497s
 656880 605679  104.99255  832  206          -    9.69313      -  90.0 18511s
 657173 606027  105.09255  888  184          -    9.69313      -  90.0 18527s
 657662 606766   12.26798   78  248          -    9.69313      -  90.0 18570s
 658403 606989   12.72591  193  178          -    9.69313      -  90.0 18579s
 658633 607382   12.72652  238  179          -    9.69313      -  90.0 18593s
 659060 607700   13.14493  367  177          -    9.69313      -  90.0 18608s
 659515 607997   14.07291  462  163          -    9.69313      -  89.9 18618s
 659834 608172   25.15717  547  130          -    9.69313      -

 695994 641610  469.79486  648  173          -    9.69696      -  90.1 20210s
 696183 641821  469.79486  654  188          -    9.69696      -  90.2 20223s
 696418 642257 infeasible  686               -    9.69696      -  90.2 20240s
 696888 642908  641.21714  805  106          -    9.69696      -  90.2 20261s
 697556 643144 infeasible  918               -    9.69696      -  90.1 20277s
 697811 643608   11.96991   58  283          -    9.69696      -  90.2 20295s
 698276 643872   12.33800  101  242          -    9.69696      -  90.2 20310s
 698566 644441   12.33800  189  225          -    9.69696      -  90.2 20334s
 698720 644441  226.76262  597  129          -    9.69696      -  90.2 20335s
 699303 644782   12.33800  261  248          -    9.69696      -  90.1 20373s
 699682 645463   12.33800  327  226          -    9.69696      -  90.1 20400s
 700404 645591  105.53369   52  219          -    9.69696      -  90.1 20411s
 700532 646035  105.95210   79  204          -    9.69696      -

AttributeError: Unable to retrieve attribute 'x'

In [4]:
import os
import psutil
process = psutil.Process(os.getpid())
mem_mb = process.memory_info().rss / 1024 / 1024
print(f"Current Memory: {mem_mb:.2f} MB")

Current Memory: 5106.25 MB
